# 6. Structured Output via Prefill + Stop Sequences

Force Claude to start its reply with a prefill (e.g. `<json>`) and halt at a stop sequence (e.g. `</json>`). Combined, this gives you a hard-bracketed payload you can parse directly — no regex, no retry loops.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from rich import print
import json

client = Anthropic()

In [2]:
def object_print(object): 
    return print(json.dumps(object.model_dump(), indent=2, default=str))

In [3]:
class ChatBot:
    """
    You are a helpful assistant
    """
    model = "claude-haiku-4-5"
    temperature = 0.7
    structured_start = ""
    structured_stop = ""
    
    def __init__(self):
        self.messages = list()
        self.system_prompt = self.__doc__.strip()
    
    def add_user_message(self, text):
        self.messages.append({"role": "user", "content": text})
    
    def add_assistant_message(self, text):
        self.messages.append({"role": "assistant", "content": text})
    
    def run(self):
        if not self.messages:
            print("How can I help you?")
            
        while True:
            user_input = input("> ")
            if user_input.lower() in ["exit", "stop", "bye", "end"]:
                break
            self.add_user_message(user_input)

            # Prefill the assistant turn to force a structured opening (e.g. ```json)
            if self.structured_start:
                self.add_assistant_message(self.structured_start)

            message = client.messages.create(
                model=self.model,
                max_tokens=1000,
                messages=self.messages,
                system=self.system_prompt,
                temperature=self.temperature,
                stop_sequences=[self.structured_stop] if self.structured_stop else None,
            )

            completion = message.content[0].text
            if self.structured_start:
                # Response continues from the prefill; replace prefill-only turn with the full text
                full_text = self.structured_start + completion
                self.messages[-1]["content"] = full_text
            else:
                full_text = completion
                self.add_assistant_message(full_text)

            print(full_text)
            return full_text

In [4]:
class JsonBot(ChatBot):
    """
    You are a json formatter.
    You can imagine any kind of object and collect its attributes and properties.
    You immediately represent it in a valid json format, no questions asked.
    Wrap the JSON in <json>...</json> tags and output nothing else.
    """
    # NOTE: prefill requires a model that supports it — Sonnet 4.6 does not.
    model = "claude-haiku-4-5"
    structured_start = "<json>"
    structured_stop = "</json>"

    def run(self):
        user_input = input("> ")
        self.add_user_message(user_input)
        # Prefill the assistant turn so the response starts inside <json>
        self.add_assistant_message(self.structured_start)

        message = client.messages.create(
            model=self.model,
            max_tokens=1000,
            messages=self.messages,
            system=self.system_prompt,
            temperature=self.temperature,
            stop_sequences=[self.structured_stop],
        )

        completion = message.content[0].text
        full_text = self.structured_start + completion
        self.messages[-1]["content"] = full_text
        # print(full_text)

        payload = full_text.removeprefix(self.structured_start).removesuffix(self.structured_stop).strip()
        return json.loads(payload)


In [5]:
chatbot = JsonBot()
chatbot.run()

>  dragon


<json>
{
  "creature": "dragon",
  "type": "mythical",
  "characteristics": {
    "body": {
      "size": "large",
      "scales": "armored",
      "wings": "membranous",
      "tail": "long and powerful"
    },
    "features": {
      "head": "reptilian with horns",
      "eyes": "glowing and intelligent",
      "teeth": "sharp and numerous",
      "claws": "razor-sharp"
    },
    "abilities": [
      "flight",
      "fire breathing",
      "treasure hoarding",
      "roaring",
      "tail whipping"
    ],
    "attributes": {
      "intelligence": "high",
      "strength": "extreme",
      "lifespan": "centuries",
      "temperament": "fierce and proud"
    }
  },
  "habitat": "mountains, caves, lairs",
  "diet": "carnivorous",
  "color_variations": [
    "red",
    "gold",
    "green",
    "black",
    "silver",
    "white"
  ],
  "cultural_significance": "symbol of power and magic across many civilizations"
}

{'creature': 'dragon',
 'type': 'mythical',
 'characteristics': {'body': {'size': 'large',
   'scales': 'armored',
   'wings': 'membranous',
   'tail': 'long and powerful'},
  'features': {'head': 'reptilian with horns',
   'eyes': 'glowing and intelligent',
   'teeth': 'sharp and numerous',
   'claws': 'razor-sharp'},
  'abilities': ['flight',
   'fire breathing',
   'treasure hoarding',
   'roaring',
   'tail whipping'],
  'attributes': {'intelligence': 'high',
   'strength': 'extreme',
   'lifespan': 'centuries',
   'temperament': 'fierce and proud'}},
 'habitat': 'mountains, caves, lairs',
 'diet': 'carnivorous',
 'color_variations': ['red', 'gold', 'green', 'black', 'silver', 'white'],
 'cultural_significance': 'symbol of power and magic across many civilizations'}